In [5]:
# ── 셀 A: KorLectureSpeech 구조 + 스키마 탐지 ────────────────
import json, random
from pathlib import Path
from collections import Counter

ROOT = Path("/data/ASR/RAW/AIHub_KorLectureSpeech")
SEED = 42
random.seed(SEED)

# 1) 상위 폴더 구조 (3단계까지)
print("=" * 70)
print("[폴더 구조 — 상위 10단계]")
def walk(d, depth=0, max_depth=10, max_items=8):
    if depth > max_depth: return
    try:
        entries = sorted(d.iterdir())
    except (PermissionError, NotADirectoryError):
        return
    dirs  = [e for e in entries if e.is_dir()]
    files = [e for e in entries if e.is_file()]
    for e in dirs[:max_items]:
        print("  " * depth + f"📁 {e.name}/")
        walk(e, depth+1, max_depth, max_items)
    if len(dirs) > max_items:
        print("  " * depth + f"   … 외 디렉터리 {len(dirs)-max_items}개")
    # 파일은 확장자별 개수만
    ext = Counter(f.suffix.lower() for f in files)
    if ext:
        print("  " * depth + f"   파일: {dict(ext)}")
walk(ROOT)

# 2) 확장자별 전체 집계 (rglob — 클 수 있으니 상한)
print("\n" + "=" * 70)
print("[전체 파일 확장자 분포] (최대 50만 스캔)")
ext_cnt = Counter()
for i, p in enumerate(ROOT.rglob("*")):
    if p.is_file():
        ext_cnt[p.suffix.lower()] += 1
    if i >= 500_000: 
        print("  (상한 도달, 일부만 집계)"); break
for k, v in ext_cnt.most_common():
    print(f"  {k or '(없음)':8s}: {v:>8,}")

# 3) JSON 한 개를 열어 실제 스키마 확인
print("\n" + "=" * 70)
print("[JSON 스키마 — 실제 배포본 확인]")
json_sample = next(ROOT.rglob("*.json"), None)
if json_sample:
    print(f"샘플: {json_sample.relative_to(ROOT)}")
    d = json.loads(json_sample.read_text(encoding="utf-8"))
    def show_keys(obj, prefix="", depth=0):
        if depth > 3: return
        if isinstance(obj, dict):
            for k, v in obj.items():
                t = type(v).__name__
                if isinstance(v, (dict, list)):
                    n = len(v)
                    print(f"  {prefix}{k} ({t}, len={n})")
                    if isinstance(v, list) and v:
                        show_keys(v[0], prefix + "  ", depth+1)
                    elif isinstance(v, dict):
                        show_keys(v, prefix + "  ", depth+1)
                else:
                    val = str(v)[:40]
                    print(f"  {prefix}{k} ({t}) = {val!r}")
    show_keys(d)
else:
    print("JSON을 찾지 못했습니다.")

# 4) txt 파일도 있는지 (가이드라인 v1.3은 .txt 분리형)
txt_sample = next(ROOT.rglob("*.txt"), None)
print(f"\n별도 .txt 전사 파일 존재: {'예 — ' + str(txt_sample.name) if txt_sample else '아니오'}")

[폴더 구조 — 상위 10단계]
📁 009.한국어_강의_데이터/
  📁 01.데이터/
    📁 1.Training/
      📁 라벨링데이터_0908_add/
        📁 KlecSpeech_train_D01_label_0/
          📁 D01/
            📁 E01/
              📁 S000001/
                   파일: {'.txt': 335, '.json': 1}
              📁 S000002/
                   파일: {'.txt': 351, '.json': 1}
              📁 S000003/
                   파일: {'.txt': 324, '.json': 1}
              📁 S000004/
                   파일: {'.txt': 282, '.json': 1}
              📁 S000005/
                   파일: {'.txt': 259, '.json': 1}
              📁 S000006/
                   파일: {'.txt': 522, '.json': 1}
              📁 S000007/
                   파일: {'.txt': 357, '.json': 1}
              📁 S000008/
                   파일: {'.txt': 159, '.json': 1}
                 … 외 디렉터리 143개
            📁 E02/
              📁 S000001/
                   파일: {'.txt': 335, '.json': 1}
              📁 S000002/
                   파일: {'.txt': 334, '.json': 1}
              📁 S000003/
                  

In [6]:
# ── 셀 B: 메타(JSON) + 실제 전사(.txt) 확인 ──────────────────
import json, random, re
from pathlib import Path
from collections import Counter

ROOT = Path("/data/ASR/RAW/AIHub_KorLectureSpeech/009.한국어_강의_데이터/01.데이터")
TRAIN_LABEL = ROOT / "1.Training" / "라벨링데이터_0908_add"
SEED = 42
random.seed(SEED)

# 세션 JSON 목록 (발화 .txt가 아니라 세션 단위 인덱스 — 수가 적어 빠름)
session_jsons = list(TRAIN_LABEL.rglob("S*.json"))
print(f"세션(강의) JSON 수: {len(session_jsons):,}")

# 카테고리 코드 → 명칭 (가이드라인 1.4.1)
CAT = {"D01":"국어","D02":"수학","D03":"사회(교과)","D04":"과학(교과)","D05":"한국사",
       "D06":"전문자격","D07":"금융","D08":"경영","D09":"IT(직업)","D10":"기술",
       "D11":"인문","D12":"철학","D13":"예술","D14":"사회(일반)","D15":"IT(일반)",
       "D16":"교양","D17":"문학","D18":"과학(일반)","D19":"교육","D99":"기타"}
def subcat_level(s):
    return {"E":"초등","M":"중등","H":"고등","G":"직업/일반"}.get(s[:1], "?")

# 세션 JSON 1개 → (메타, 발화 txt 경로 리스트)
def load_session(jp):
    d = json.loads(jp.read_text(encoding="utf-8"))
    ds = d.get("dataSet", {})
    ti = ds.get("typeInfo", {})
    spk = (ti.get("speakers") or [{}])[0]
    meta = {
        "category": ti.get("category"), "subcategory": ti.get("subcategory"),
        "place": ti.get("place"), "inputType": ti.get("inputType"),
        "gender": spk.get("gender"), "age": spk.get("age"),
        "residence": spk.get("residence"), "date": ds.get("date"),
        "n_dialogs": len(ds.get("dialogs", [])),
    }
    # textPath(가상경로)를 실제 .txt로: 같은 세션 폴더 안의 .txt를 stem으로
    sess_dir = jp.parent
    txts = sorted(sess_dir.glob("*.txt"))
    return meta, txts, jp

# 전사 .txt 읽기
def read_txt(p):
    try:
        return p.read_text(encoding="utf-8").strip()
    except Exception:
        return p.read_text(encoding="cp949", errors="replace").strip()

# 랜덤 세션 몇 개에서 전사 샘플 출력
print("\n" + "="*80)
print("[실제 전사 .txt 샘플]")
sample_sessions = random.sample(session_jsons, 6)
for jp in sample_sessions:
    meta, txts, _ = load_session(jp)
    cat_code = jp.parts[[i for i,x in enumerate(jp.parts) if x.startswith("D") and len(x)==3][-1]]
    print(f"\n■ {jp.parent.name} | {CAT.get(cat_code,'?')}({cat_code}) "
          f"{subcat_level(jp.parent.parent.name)}({jp.parent.parent.name}) | "
          f"{meta['gender']}/{meta['age']} | {meta['place']}/{meta['inputType']} | "
          f"발화 {meta['n_dialogs']}개")
    for tp in txts[:4]:
        print(f"   [{tp.stem}] {read_txt(tp)[:90]}")

# @ 와 특수 태그를 곧장 찾아보기 (작은 표본으로 빠르게 존재 확인)
print("\n" + "="*80)
print("[특수표기 빠른 탐지 — @ 및 잡음/이중전사 태그]")
probe_txts = []
for jp in random.sample(session_jsons, 40):
    probe_txts += sorted(jp.parent.glob("*.txt"))[:30]
print(f"탐지 표본 .txt: {len(probe_txts):,}")

pat = {
    "@": re.compile(r"@"),
    "이중전사 (..)/(..)": re.compile(r"\)/\("),
    "잡음 b/": re.compile(r"(?:^|\s)b/"),
    "잡음 l/": re.compile(r"(?:^|\s)l/"),
    "잡음 o/": re.compile(r"(?:^|\s)o/"),
    "잡음 n/": re.compile(r"(?:^|\s)n/"),
    "불명 u/": re.compile(r"(?:^|\s)u/"),
    "불명 어절*": re.compile(r"\*"),
    "반복 어절+": re.compile(r"\+"),
}
hits = Counter()
ex = {k: [] for k in pat}
for tp in probe_txts:
    t = read_txt(tp)
    for k, rgx in pat.items():
        if rgx.search(t):
            hits[k] += 1
            if len(ex[k]) < 3:
                ex[k].append(t[:85])

for k in pat:
    print(f"\n── {k}: {hits[k]}/{len(probe_txts)} 파일")
    for e in ex[k]:
        print(f"     {e}")

세션(강의) JSON 수: 7,942

[실제 전사 .txt 샘플]

■ S001012 | 교양(D16) 직업/일반(G02) | 여/30 | studio/broadcast | 발화 252개
   [000000] n/ 한 주일 동안 안녕하셨습니까?
   [000001] n/ 한 (2)/(이) (3년)/(삼 년) 전의 일로 기억이 되는데요. 미국 그/ 로스앤젤레스 타임즈 라는 신문의 세계 (41개국)/(마흔 한 개국)의 여성의 지
   [000002] 그 중에서 이/ 한국의 여성들의 지위 교육 수준이라든가 생활 수준이라든가 사회적인 여건이라든가 여러 가지를 고려해서 매긴 순위가 그/ (41개)/(마흔 한 개) 
   [000003] 참 충격을 저는 받았었습니다. 요즈음의 그/ 히트하는 대중가요 중에 (super man)/(수퍼맨)이라는 그런 노래가 있지요.

■ S000022 | 사회(교과)(D03) 중등(M03) | 여/20(?) | studio/broadcast | 발화 422개
   [000000] 안녕하세요 친구들 뉴런 사회 이 수업의
   [000001] 입니다 어서 오세요. 자/ 오,
   [000002] 오늘도 우리 힘차게 한번 사회 재미난 길로 달려가 보도록 하겠습니다 오늘 배워 볼 내용은요?
   [000003] 계속해서 경제생활과 어/

■ S000199 | 수학(D02) 직업/일반(G01) | 남/30(?) | studio/broadcast | 발화 541개
   [000000] n/ 여러분 안녕하세요. 오늘은 그래프와 관계에 대해서 (1번)/(한 번) 공부를 해 보겠습니다.
   [000001] n/ 그래프랑 관계가 무슨 소리지?
   [000002] n/ 그렇죠. 여러분들이 만나기 힘든 파트가 있는데요.
   [000003] n/ 전국에 그 (14만명)/(십 사만 명) 정도 가요,

■ S000433 | 수학(D02) 초등(E06) | 여/30(?) | studio/broadcast | 발화 163개
   [000000] n/ 전국의 사랑하

In [7]:
# ── 셀: @/ 인명 마커 정밀 분석 (비식별화 여부 판정) ──────────
import re, random
from pathlib import Path
from collections import Counter

ROOT = Path("/data/ASR/RAW/AIHub_KorLectureSpeech/009.한국어_강의_데이터/01.데이터")
TRAIN_LABEL = ROOT / "1.Training" / "라벨링데이터_0908_add"
SEED = 42
random.seed(SEED)

def read_txt(p):
    try:    return p.read_text(encoding="utf-8").strip()
    except Exception: return p.read_text(encoding="cp949", errors="replace").strip()

# @/뒤에 오는 토큰(이름) 추출: '@/' 직후 공백 전까지
re_at = re.compile(r"@/(\S+)")

# 표본을 키워서 @ 사례를 최대한 모음 (@ 자체가 희소하므로 세션 많이 스캔)
session_jsons = list(TRAIN_LABEL.rglob("S*.json"))
sample_sessions = random.sample(session_jsons, min(800, len(session_jsons)))

names = Counter()           # @/뒤 토큰 전체
second_char = Counter()     # 이름 둘째 글자 분포 (별 가설 검증)
first_char = Counter()      # 첫 글자(성) 분포
name_examples = []
files_with_at = 0
total_files = 0

for jp in sample_sessions:
    for tp in jp.parent.glob("*.txt"):
        total_files += 1
        t = read_txt(tp)
        found = re_at.findall(t)
        if found:
            files_with_at += 1
            if len(name_examples) < 25:
                # @ 주변 문맥 일부 저장
                idx = t.find("@/")
                name_examples.append(t[max(0,idx-15):idx+40])
        for nm in found:
            # 뒤따르는 문장부호 제거
            nm = re.sub(r"[.,?!]+$", "", nm)
            names[nm] += 1
            if len(nm) >= 2:
                first_char[nm[0]] += 1
                second_char[nm[1]] += 1

print(f"스캔 .txt: {total_files:,} / @ 포함 파일: {files_with_at}")
print(f"@/이름 토큰 총 출현: {sum(names.values())} / 고유: {len(names)}\n")

print("[@/뒤 이름 둘째 글자 분포] — '별'에 쏠리면 비식별화 마스킹 강력 시사")
for ch, c in second_char.most_common(15):
    print(f"  '{ch}' : {c}")

print("\n[@/뒤 이름 첫 글자(성?) 분포 상위]")
for ch, c in first_char.most_common(15):
    print(f"  '{ch}' : {c}")

print("\n[@/이름 토큰 빈도 상위 30]")
for nm, c in names.most_common(30):
    print(f"  {nm} : {c}")

print("\n[문맥 예시]")
for ex in name_examples:
    print(f"   …{ex}…")

스캔 .txt: 242,724 / @ 포함 파일: 66
@/이름 토큰 총 출현: 103 / 고유: 70

[@/뒤 이름 둘째 글자 분포] — '별'에 쏠리면 비식별화 마스킹 강력 시사
  '별' : 23
  '기' : 10
  'W' : 7
  '나' : 7
  '신' : 7
  '안' : 4
  'J' : 3
  '이' : 3
  '적' : 3
  '현' : 2
  '도' : 2
  '리' : 2
  '관' : 2
  '진' : 2
  '승' : 2

[@/뒤 이름 첫 글자(성?) 분포 상위]
  '(' : 13
  '슬' : 10
  '한' : 9
  '김' : 8
  '이' : 7
  '박' : 7
  '욱' : 7
  '안' : 5
  '배' : 4
  '최' : 4
  '정' : 3
  '강' : 2
  '윤' : 2
  '남' : 1
  '임' : 1

[@/이름 토큰 빈도 상위 30]
  슬기 : 10
  욱신 : 7
  (William : 5
  한나 : 5
  (John : 3
  배안나 : 3
  최적 : 3
  이별민 : 2
  안도히로시게 : 2
  (Webb)/(웨브) : 2
  안나 : 2
  이별은 : 1
  남별경 : 1
  정별 : 1
  임별경 : 1
  박별경 : 1
  박별 : 1
  한별희 : 1
  한별라 : 1
  조별주 : 1
  안별아 : 1
  공별우 : 1
  김별영 : 1
  박별리 : 1
  김별진 : 1
  강현태 : 1
  페리스브랑먼 : 1
  (Edward : 1
  고드윈 : 1
  (Samuel : 1

[문맥 예시]
   …@/이별은 친구 @/남별경 친구 @/정별 친구 @/임별경 친구 어/ (H…
   …우리 @/이별민 친구 이름 불러 달라고 했는데 이거 맞혀보세요. 친구 법률은 …
   …그렇습니다. @/박별경 친구가 굉장히 자세하게 잘 썼어요. 그저 국가의 운영 원리라든…
   …자/ @/박별 친구 @/한별희 친구 @/한별라 자/ 그래요 우리 많은 친구들이…
   …@ 권별은 친구 @/조별주 

In [8]:
# ── 셀: 개인정보(PII) 패턴 전수 점검 — @ 외 모든 유형 ────────
import re, random
from pathlib import Path
from collections import Counter, defaultdict

ROOT = Path("/data/ASR/RAW/AIHub_KorLectureSpeech/009.한국어_강의_데이터/01.데이터")
TRAIN_LABEL = ROOT / "1.Training" / "라벨링데이터_0908_add"
SEED = 42
random.seed(SEED)

def read_txt(p):
    try:    return p.read_text(encoding="utf-8").strip()
    except Exception: return p.read_text(encoding="cp949", errors="replace").strip()

# 이중전사 괄호를 먼저 벗겨낸 '순수 텍스트'에서 패턴을 봐야 오탐이 적음
re_dual = re.compile(r"\(([^)]*)\)/\(([^)]*)\)")   # (철자)/(발음) → 철자만 남김
def strip_dual(t):
    return re_dual.sub(lambda m: m.group(1), t)

# PII 후보 패턴들
PII = {
    "전화번호(하이픈)":   re.compile(r"\d{2,4}[-\s]\d{3,4}[-\s]\d{4}"),
    "휴대폰 010":         re.compile(r"010[-\s]?\d{3,4}[-\s]?\d{4}"),
    "이메일":             re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"),
    "골뱅이(한글이메일)": re.compile(r"골뱅이"),
    "URL":                re.compile(r"(?:https?://|www\.)\S+"),
    "주민번호류":         re.compile(r"\d{6}[-\s]?\d{7}"),
    "계좌/카드 긴숫자":   re.compile(r"\d{4}[-\s]\d{4}[-\s]\d{4}"),
    "주소(동/번지/호)":   re.compile(r"[가-힣]+(동|로|길)\s?\d+(번지|호)?"),
    "비밀번호 언급":      re.compile(r"비밀번호|패스워드|비번"),
    "아이디 언급":        re.compile(r"아이디|계정|로그인"),
    "@마커":              re.compile(r"@/?\S+"),
}

session_jsons = list(TRAIN_LABEL.rglob("S*.json"))
sample_sessions = random.sample(session_jsons, min(1500, len(session_jsons)))

hits = Counter()
examples = defaultdict(list)
total = 0

for jp in sample_sessions:
    for tp in jp.parent.glob("*.txt"):
        total += 1
        raw = read_txt(tp)
        t = strip_dual(raw)        # 이중전사 철자측만 남긴 텍스트
        for name, rgx in PII.items():
            m = rgx.search(t)
            if m:
                hits[name] += 1
                if len(examples[name]) < 5:
                    s = max(0, m.start()-20)
                    examples[name].append(t[s:m.end()+25].replace("\n"," "))

print(f"스캔 .txt: {total:,}\n")
print("[PII 패턴 적중 — 전사에 실제 개인정보가 새어있는지]")
for name in PII:
    c = hits[name]
    flag = "" if c == 0 else ("  ← 확인필요" if name not in ("@마커",) else "")
    print(f"  {name:18s}: {c:>5,} 파일 ({c/max(total,1)*100:5.2f}%){flag}")

print("\n[적중 예시]")
for name in PII:
    if examples[name]:
        print(f"\n── {name} ──")
        for ex in examples[name]:
            print(f"   …{ex}…")

스캔 .txt: 444,287

[PII 패턴 적중 — 전사에 실제 개인정보가 새어있는지]
  전화번호(하이픈)         :    19 파일 ( 0.00%)  ← 확인필요
  휴대폰 010           :     0 파일 ( 0.00%)
  이메일               :     0 파일 ( 0.00%)
  골뱅이(한글이메일)        :     8 파일 ( 0.00%)  ← 확인필요
  URL               :     0 파일 ( 0.00%)
  주민번호류             :    56 파일 ( 0.01%)  ← 확인필요
  계좌/카드 긴숫자         :     2 파일 ( 0.00%)  ← 확인필요
  주소(동/번지/호)        : 2,796 파일 ( 0.63%)  ← 확인필요
  비밀번호 언급           :    30 파일 ( 0.01%)  ← 확인필요
  아이디 언급            :   314 파일 ( 0.07%)  ← 확인필요
  @마커               :   775 파일 ( 0.17%)

[적중 예시]

── 전화번호(하이픈) ──
   …수 있다 이 얘기는 순위와 관계없이 3400 2700 2000 만원은 1700은 순위와 관계에 무조건.…
   … 24 6 5 30. 그러면 분모를 10 100 1000으로 만들어줘야 돼요. 4의 짝꿍 외워두면 편…
   …1 10 100 1000 10000…
   …한 포인트 뭐야. 유한소수는 분모가 10 100 1000 10000 10의 거듭제곱이 유한소수입니다.…
   …렇게 많아요. 459 460 461 462 463 464465번 한번 여러분 먼저 쭉/ 한번 풀어보시면…

── 골뱅이(한글이메일) ──
   …측 상단에 보시면 지난주에 그러셨던 골뱅이 마크를 이용한 어떠 그/ 필터효과, 블러처리…
   …하시게 되면 이런 형태로 그림 있는 골뱅이 부분이 검정색으로 채워진 것을 알 수가 있습…
   …이 좌측 상단에 있는 골뱅이 마크를 조금

In [9]:
# ── 셀: @ 뒤 토큰 전수 조회 (train 전체) ─────────────────────
import re
from pathlib import Path
from collections import Counter

ROOT = Path("/data/ASR/RAW/AIHub_KorLectureSpeech/009.한국어_강의_데이터/01.데이터")
TRAIN_LABEL = ROOT / "1.Training" / "라벨링데이터_0908_add"

def read_txt(p):
    try:    return p.read_text(encoding="utf-8").strip()
    except Exception: return p.read_text(encoding="cp949", errors="replace").strip()

# @ 바로 뒤(슬래시 있든 없든)에 오는 토큰을 잡고, 동시에 @ 포함 문장 전체도 보관
re_after = re.compile(r"@/?\s*(\S+)")     # @/슬기, @ 권별은, @슬기 모두 포착

after_tokens = Counter()
contexts = []          # @ 포함 전사 원문 (중복 제거용 set 후 정렬)
n_at_files = 0
n_at_total = 0

# 라벨 트리 전체의 .txt를 직접 순회 (json 거치지 않고 바로 .txt)
for tp in TRAIN_LABEL.rglob("*.txt"):
    t = read_txt(tp)
    if "@" not in t:
        continue
    n_at_files += 1
    for m in re_after.finditer(t):
        tok = re.sub(r"[.,?!]+$", "", m.group(1))   # 뒤 문장부호 제거
        after_tokens[tok] += 1
        n_at_total += 1
    contexts.append((tp.relative_to(TRAIN_LABEL), t))

print(f"@ 포함 .txt 파일: {n_at_files}")
print(f"@ 뒤 토큰 총 출현: {n_at_total} / 고유: {len(after_tokens)}\n")

print("[@ 뒤 토큰 — 전체 목록 (빈도순)]")
for tok, c in after_tokens.most_common():
    print(f"  {c:>3}  {tok}")

print("\n" + "="*80)
print("[@ 포함 전사 원문 — 전체]")
for rel, t in contexts:
    print(f"\n· {rel}")
    print(f"  {t}")

@ 포함 .txt 파일: 3927
@ 뒤 토큰 총 출현: 5246 / 고유: 2977

[@ 뒤 토큰 — 전체 목록 (빈도순)]
   73  미켈란젤로
   44  크세노폰
   41  왕건
   38  키루스
   31  옥희
   27  알키비아데스
   23  카라바조
   23  김일성
   20  보미
   18  유방
   16  티치아노
   16  크세르크세스
   15  유방은
   15  전봉준
   13  키케로
   13  최태성
   13  욱신
   13  경업이
   12  이별민
   12  옥희야
   12  류성완
   12  브루넬레스키
   12  민수
   11  김별은
   11  이별은
   11  김별현
   11  까라바조
   11  장보고
   11  파울클레
   11  현지
   10  류성완입니다
   10  리쿠르고스
   10  크로이소스
   10  슬기
   10  최치원
   10  로트렉
    9  흄
    9  화이트헤드
    9  페트라르카
    9  윤
    9  윤동주의
    9  리버먼
    9  백남준
    9  박술희
    8  항우
    8  서근철
    8  최적
    8  한나
    8  김별희
    8  동민
    8  문기는
    8  스틸리코
    8  미켈로쪼
    8  서태지
    8  듀나
    8  견훤
    8  김구
    7  나훈아
    7  장량
    7  캐번디시
    7  김별민
    7  김별윤
    7  점순이
    7  흥부가
    7  라파엘로
    7  도나텔로
    7  엘그레코
    7  테미스토클레스
    7  헤로도토스
    7  윤용규는
    7  지혜
    7  유관순
    7  민석
    6  쿤
    6  조조
    6  김별영
    6  만도와
    6  진수는
    6  점순이가
    6  김희진
    6  수진
    6  미켈란젤로가
    6  